# Credit memo walkthrough: Microsoft Corporation

Vanuja Ekanayake, BBA Finance candidate, Baruch College Zicklin School of Business.

This note walks through one issuer with the same definitions used for the rest of the file. The amounts are Form 10-K figures from the SEC companyfacts cache. The notebook does not call EDGAR.

Fiscal 2025, the year ended June 30, 2025, is the year tied to the face of the Form 10-K filed July 30, 2025. The scorecard recommendation at the end of the note uses the latest year on file, fiscal 2026.

The definitions, stated once in `src/ratios.py`:

- EBITDA is operating income plus depreciation and amortization.
- Total debt is short-term borrowings plus the current portion of long-term debt plus long-term debt. Operating leases and finance leases are excluded.
- Net debt is total debt minus cash and cash equivalents. Short-term investments are not deducted.
- Debt / EBITDA and EBITDA / Interest use those amounts. A leverage multiple is not meaningful when EBITDA is not positive.


In [1]:
from pathlib import Path
import pandas as pd

pd.set_option("display.float_format", lambda value: f"{value:,.2f}")
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

statements = pd.read_csv(Path("data/processed/annual_statements.csv"))
scorecard = pd.read_csv(Path("data/processed/scorecard.csv"))
microsoft = statements.loc[statements["ticker"] == "MSFT"].sort_values("fiscal_year")
columns = [
    "fiscal_year", "period_end", "revenue", "operating_income",
    "depreciation_and_amortization", "ebitda", "short_term_borrowings",
    "current_portion_long_term_debt", "long_term_debt", "total_debt",
    "interest", "debt_to_ebitda", "ebitda_to_interest",
]
microsoft[columns]


,fiscal_year,period_end,revenue,operating_income,depreciation_and_amortization,ebitda,short_term_borrowings,current_portion_long_term_debt,long_term_debt,total_debt,interest,debt_to_ebitda,ebitda_to_interest
15,2022,2022-06-30,"198,270,000,000.00","83,383,000,000.00","14,600,000,000.00","97,983,000,000.00",0.00,"2,749,000,000.00","47,032,000,000.00","49,781,000,000.00","2,063,000,000.00",0.51,47.50
16,2023,2023-06-30,"211,915,000,000.00","88,523,000,000.00","13,500,000,000.00","102,023,000,000.00",0.00,"5,247,000,000.00","41,990,000,000.00","47,237,000,000.00","1,968,000,000.00",0.46,51.84
17,2024,2024-06-30,"245,122,000,000.00","109,433,000,000.00","20,000,000,000.00","129,433,000,000.00","6,693,000,000.00","2,249,000,000.00","42,688,000,000.00","51,630,000,000.00","2,935,000,000.00",0.40,44.10
18,2025,2025-06-30,"281,724,000,000.00","128,528,000,000.00","28,000,000,000.00","156,528,000,000.00",0.00,"2,999,000,000.00","40,152,000,000.00","43,151,000,000.00","2,385,000,000.00",0.28,65.63
19,2026,2026-06-30,"331,839,000,000.00","155,237,000,000.00","39,000,000,000.00","194,237,000,000.00",0.00,"9,227,000,000.00","31,067,000,000.00","40,294,000,000.00","3,051,000,000.00",0.21,63.66


## Fiscal 2025, tied to the Form 10-K

The face of the fiscal 2025 Form 10-K, filed July 30, 2025, reports the amounts in the `face` column. The model column is the companyfacts extract. Debt / EBITDA and EBITDA / Interest are then computed from the tied amounts, not typed in.

Filing: https://www.sec.gov/Archives/edgar/data/789019/000095017025100235/msft-20250630.htm


In [2]:
face = {
    "revenue": 281_724_000_000,
    "operating_income": 128_528_000_000,
    "depreciation_and_amortization": 28_000_000_000,
    "short_term_borrowings": 0,
    "current_portion_long_term_debt": 2_999_000_000,
    "long_term_debt": 40_152_000_000,
    "total_debt": 43_151_000_000,
    "interest": 2_385_000_000,
    "ebitda": 156_528_000_000,
}
year = microsoft.loc[microsoft["fiscal_year"] == 2025].iloc[0]
rows = []
for field, filing_amount in face.items():
    model_amount = float(year[field])
    rows.append({
        "item": field,
        "model": model_amount,
        "form_10k": filing_amount,
        "difference": model_amount - filing_amount,
    })
tie = pd.DataFrame(rows)
assert (tie["difference"].abs() < 1).all()
ebitda = year["operating_income"] + year["depreciation_and_amortization"]
total_debt = (
    year["short_term_borrowings"]
    + year["current_portion_long_term_debt"]
    + year["long_term_debt"]
)
debt_to_ebitda = total_debt / ebitda
coverage = ebitda / year["interest"]
print(f"EBITDA = {year['operating_income']:,.0f} + {year['depreciation_and_amortization']:,.0f} = {ebitda:,.0f}")
print(f"Total debt = {year['short_term_borrowings']:,.0f} + {year['current_portion_long_term_debt']:,.0f} + {year['long_term_debt']:,.0f} = {total_debt:,.0f}")
print(f"Debt / EBITDA = {total_debt:,.0f} / {ebitda:,.0f} = {debt_to_ebitda:.4f}x")
print(f"EBITDA / Interest = {ebitda:,.0f} / {year['interest']:,.0f} = {coverage:.2f}x")
tie


EBITDA = 128,528,000,000 + 28,000,000,000 = 156,528,000,000
Total debt = 0 + 2,999,000,000 + 40,152,000,000 = 43,151,000,000
Debt / EBITDA = 43,151,000,000 / 156,528,000,000 = 0.2757x
EBITDA / Interest = 156,528,000,000 / 2,385,000,000 = 65.63x


,item,model,form_10k,difference
0,revenue,"281,724,000,000.00",281724000000,0.00
1,operating_income,"128,528,000,000.00",128528000000,0.00
2,depreciation_and_amortization,"28,000,000,000.00",28000000000,0.00
3,short_term_borrowings,0.00,0,0.00
4,current_portion_long_term_debt,"2,999,000,000.00",2999000000,0.00
5,long_term_debt,"40,152,000,000.00",40152000000,0.00
6,total_debt,"43,151,000,000.00",43151000000,0.00
7,interest,"2,385,000,000.00",2385000000,0.00
8,ebitda,"156,528,000,000.00",156528000000,0.00


## How to read fiscal 2025

Debt of $43.2 billion against EBITDA of $156.5 billion is 0.28x. Interest of $2.4 billion is covered 65.63x. Those two results are what a strong credit looks like on this scale. They are not a rating.

Two items sit outside the ratios on purpose.

Finance lease liabilities are tagged separately from long-term debt and are excluded. At June 30, 2025 they were large relative to the $43.2 billion of funded debt. A lease-adjusted leverage measure would be higher. Operating lease liabilities are also excluded.

Net debt deducts cash of $30.2 billion and does not deduct short-term investments. The investment portfolio is large. Net debt of about $12.9 billion is not the same statement as "net cash" once the investments are counted, and it is not a weak result either. The scorecard follows the stated definition and shows the investments beside it.


In [3]:
year_2025 = microsoft.loc[microsoft["fiscal_year"] == 2025].iloc[0]
print(f"Cash: {year_2025['cash']:,.0f}")
print(f"Short-term investments, not in net debt: {year_2025['short_term_investments']:,.0f}")
print(f"Net debt: {year_2025['net_debt']:,.0f}")
print(f"Finance lease liability, excluded: {year_2025['finance_lease_liability']:,.0f}")
print(f"Operating lease liability, excluded: {year_2025['operating_lease_liability']:,.0f}")


Cash: 30,242,000,000
Short-term investments, not in net debt: 64,323,000,000
Net debt: 12,909,000,000
Finance lease liability, excluded: 46,172,000,000
Operating lease liability, excluded: 22,861,000,000


## Latest year on file

The recommendation uses fiscal 2026, the Form 10-K filed July 29, 2026. The arithmetic is the same. The current ratio is the only line outside the top bucket. It is 1.23x, which the published scale calls Acceptable, and it carries a 5 percent weight. Everything else in the scorecard is in the top bucket. The composite is 3.95, which maps to Strong.


In [4]:
latest = scorecard.loc[(scorecard["ticker"] == "MSFT") & (scorecard["fiscal_year"] == 2026)].copy()
show = latest[["ratio", "value", "published_weight", "points", "bucket", "weighted_points", "note"]]
composite = float(latest["composite"].iloc[0])
category = latest["category"].iloc[0]
print(f"Composite {composite:.2f}: {category}")
show


Composite 3.95: Strong


,ratio,value,published_weight,points,bucket,weighted_points,note
0,Debt / EBITDA,0.21,0.20,4,Strong,0.80,Scored on the published scale.
1,Net debt / EBITDA,0.10,0.15,4,Strong,0.60,Scored on the published scale.
2,EBITDA / Interest,63.66,0.20,4,Strong,0.80,Scored on the published scale.
3,(EBITDA - capex) / Interest,25.66,0.10,4,Strong,0.40,Scored on the published scale.
4,CFO / Debt,4.54,0.15,4,Strong,0.60,Scored on the published scale.
5,FCF / Debt,1.66,0.05,4,Strong,0.20,Scored on the published scale.
6,Debt / capitalization,0.08,0.10,4,Strong,0.40,Scored on the published scale.
7,Current ratio,1.23,0.05,3,Acceptable,0.15,Scored on the published scale.


## Conclusion

Microsoft is the strong end of this five-company file. Funded leverage is a fraction of a turn and interest coverage is very high in both fiscal 2025 and fiscal 2026. The current ratio does not change that conclusion at a 5 percent weight.

The conclusion is narrow. It uses unadjusted EBITDA, it leaves leases out of debt, and it does not deduct the short-term investment portfolio. Those choices are in the ratio definitions. They are not hidden adjustments.
